 # RISS 학술논문 검색 및 다운로드 실습
 - [실습 페이지](https://www.riss.kr/search/Search.do?isDetailSearch=N&searchGubun=true&viewYn=OP&query=%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5&queryText=&iStartCount=0&iGroupView=5&icate=all&colName=re_a_kor&exQuery=&exQueryText=&order=%2FDESC&onHanja=false&strSort=RANK&pageScale=10&orderBy=&fsearchMethod=search&isFDetailSearch=N&sflag=1&searchQuery=%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5&fsearchSort=&fsearchOrder=&limiterList=&limiterListText=&facetList=&facetListText=&fsearchDB=&resultKeyword=%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5&pageNumber=1&p_year1=&p_year2=&dorg_storage=&mat_type=&mat_subtype=&fulltext_kind=&t_gubun=&learning_type=&language_code=&ccl_code=&language=&inside_outside=&fric_yn=&db_type=&image_yn=&regnm=&gubun=&kdc=&ttsUseYn=)


논문분석 -> 연구트랜드, 정부가 현재 많이 투자하는 분야는?

 ## 1. 환경 설정 및 드라이버 함수 정의

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import urllib.parse
import time
import os

def get_chrome_driver_with_download(download_path):
    """
    파일 다운로드 경로가 설정된 Chrome 드라이버 생성
    
    Args:
        download_path: 파일을 저장할 디렉토리 경로
    
    Returns:
        WebDriver: 설정된 Chrome 드라이버
    """
    chrome_options = Options()
    
    # ========== 기본 옵션 ==========
    chrome_options.add_argument('--start-maximized')
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    
    # ========== 자동화 감지 회피 ==========
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    
    # ========== User-Agent 설정 ==========
    user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    chrome_options.add_argument(f'user-agent={user_agent}')
    
    # ========== 다운로드 경로 설정 ==========
    # 다운로드 관련 환경 설정
    chrome_options.add_experimental_option("prefs", {
        "download.default_directory": download_path,  # 다운로드 저장 경로
        "download.prompt_for_download": False,  # 다운로드 시 대화상자 표시 안 함
        "download.directory_upgrade": True,  # 다운로드 설정 업그레이드
        "safebrowsing.enabled": True  # 안전 브라우징 활성화
    })
    
    # ========== 드라이버 생성 ==========
    driver = webdriver.Chrome(options=chrome_options)
    
    # ========== JavaScript를 통한 추가 위장 ==========
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': '''
            Object.defineProperty(navigator, 'webdriver', {
                get: () => undefined
            })
        '''
    })
    
    return driver

print("드라이버 함수 정의 완료!")

드라이버 함수 정의 완료!


 ## 2. 다운로드 경로 설정 및 드라이버 생성

In [2]:
# 현재 작업 디렉토리의 절대 경로 가져오기
current_directory = os.path.abspath(os.getcwd())

# 다운로드 저장 폴더 경로 생성
download_folder = os.path.join(current_directory, '논문목록')

# 폴더가 없으면 생성
if not os.path.exists(download_folder):
    os.makedirs(download_folder)
    print(f"다운로드 폴더 생성: {download_folder}")
else:
    print(f"다운로드 폴더 확인: {download_folder}")

# 드라이버 생성
driver = get_chrome_driver_with_download(download_folder)

# WebDriverWait 객체 생성 (최대 10초 대기)
wait = WebDriverWait(driver, 10)

print("드라이버 생성 완료!")

다운로드 폴더 확인: /Users/t2023-m0099/Documents/NBC_Data_9th/설무아T/데이터 수집/4회차세션/논문목록
드라이버 생성 완료!


 ## 3. 검색어 설정 및 URL 생성



 **실습 포인트:**

 - RISS 검색 URL 구조 이해

 - `urllib.parse.quote()`를 사용한 한글 URL 인코딩

 - 페이지네이션을 위한 `iStartCount` 파라미터

In [3]:
import urllib.parse

def create_riss_url(search_keyword, start_count=0):
    """
    RISS 검색 URL을 생성하는 함수
    
    Args:
        search_keyword: 검색할 키워드
        start_count: 시작 인덱스 (0, 100, 200, ...) 기본값 0
    
    Returns:
        str: 완성된 검색 URL
    """
    # RISS 기본 검색 URL
    service_url = (
        "http://www.riss.kr/search/Search.do?"
        "isDetailSearch=N&searchGubun=true&viewYn=OP&queryText=&exQuery=&exQueryText=&"
        "order=%2FDESC&onHanja=false&strSort=RANK&p_year1=&p_year2=&orderBy=&"
        "mat_type=&mat_subtype=&fulltext_kind=&t_gubun=&learning_type=&ccl_code=&"
        "inside_outside=&fric_yn=&image_yn=&gubun=&kdc=&ttsUseYn=&l_sub_code=&"
        "fsearchMethod=&sflag=1&isFDetailSearch=N&pageNumber=1&resultKeyword=&"
        "fsearchSort=&fsearchOrder=&limiterList=&limiterListText=&facetList=&facetListText=&"
        "fsearchDB=&icate=re_a_kor&colName=re_a_kor&pageScale=100&isTab=Y&"
        "regnm=&dorg_storage=&language=&language_code=&clickKeyword=&relationKeyword="
    )
    
    # 검색어 파라미터 추가 (한글 UTF-8 인코딩)
    # 실제로 url을 테스트해보며 확인해야해. (gpt가 정확하게 알 수 없는 정보)
    param = f"&strQuery={urllib.parse.quote(search_keyword)}"   # 검색 키워드
    param += f"&query={urllib.parse.quote(search_keyword)}" # 검색 키워드
    param += f"&iStartCount={start_count}"  # 페이지 정보
    
    # 최종 URL 생성
    url = service_url + param
    
    return url


# 사용 예시
search_term = "인공지능"

# 첫 번째 페이지 (0~99번 논문)
url1 = create_riss_url(search_term, 0)
print(f"1페이지 URL 생성 완료")
print(f"URL 확인: {url1}...\n")

1페이지 URL 생성 완료
URL 확인: http://www.riss.kr/search/Search.do?isDetailSearch=N&searchGubun=true&viewYn=OP&queryText=&exQuery=&exQueryText=&order=%2FDESC&onHanja=false&strSort=RANK&p_year1=&p_year2=&orderBy=&mat_type=&mat_subtype=&fulltext_kind=&t_gubun=&learning_type=&ccl_code=&inside_outside=&fric_yn=&image_yn=&gubun=&kdc=&ttsUseYn=&l_sub_code=&fsearchMethod=&sflag=1&isFDetailSearch=N&pageNumber=1&resultKeyword=&fsearchSort=&fsearchOrder=&limiterList=&limiterListText=&facetList=&facetListText=&fsearchDB=&icate=re_a_kor&colName=re_a_kor&pageScale=100&isTab=Y&regnm=&dorg_storage=&language=&language_code=&clickKeyword=&relationKeyword=&strQuery=%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5&query=%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5&iStartCount=0...



 ## 4. RISS 검색 페이지 접속

In [4]:
# 첫 번째 페이지 URL 생성
url = create_riss_url(search_term, 0)

# 페이지 접속
driver.get(url)

# 페이지가 완전히 로드될 때까지 대기
time.sleep(3)

print(f"RISS 검색 페이지 접속 완료")
print(f"검색어: {search_term}")
print(f"페이지 제목: {driver.title}")

RISS 검색 페이지 접속 완료
검색어: 인공지능
페이지 제목: RISS 검색 - 국내학술지논문


 ## 5. 전체 선택 체크박스 클릭



 **실습 포인트:**

 - 체크박스 요소 찾기

 - `WebDriverWait`를 사용한 요소 로딩 대기

 - 체크박스 클릭

In [5]:
# 전체 선택 체크박스 요소가 나타날 때까지 대기
# XPath를 사용하여 요소 찾기
select_all_checkbox = wait.until(
    EC.presence_of_element_located((By.XPATH, 
        '//*[@id="divContent"]/div/div[2]/div/div[3]/div[1]/div[1]/label/span'))
)

# 체크박스 클릭
select_all_checkbox.click()

print("전체 선택 체크박스 클릭 완료")

# 체크박스가 선택되었는지 확인하기 위한 짧은 대기
time.sleep(1)

전체 선택 체크박스 클릭 완료


 ## 6. 내보내기 버튼 클릭 (JavaScript 함수 실행)



 **실습 포인트:**

 - 웹페이지의 JavaScript 함수 직접 실행

 - `execute_script()` 메서드 사용

In [6]:
# RISS 웹사이트의 exportData() JavaScript 함수 실행
# 이 함수는 선택된 논문 목록을 내보내기 위한 팝업을 엽니다
driver.execute_script("exportData()")

print("내보내기 JavaScript 함수 실행 완료")

# 새 창(팝업)이 열릴 때까지 대기
time.sleep(2)

내보내기 JavaScript 함수 실행 완료


 ## 7. 새 창(팝업) 핸들링



 **중요 개념: 창(Window) 핸들링**



 Selenium에서는 각 브라우저 창이나 탭을 "window handle"로 구분합니다.

 - `driver.window_handles`: 현재 열린 모든 창의 핸들 리스트

 - `driver.window_handles[0]`: 첫 번째 창 (메인 창)

 - `driver.window_handles[1]`: 두 번째 창 (팝업 창)

 - `driver.switch_to.window()`: 특정 창으로 전환

In [7]:
# 현재 열린 창의 개수 확인
print(f"현재 열린 창의 개수: {len(driver.window_handles)}")

# 각 창의 핸들 출력
for i, handle in enumerate(driver.window_handles):
    print(f"창 {i}: {handle}")

# 새 창(팝업 창)으로 전환
# window_handles[1]이 내보내기 팝업창
# switch_to.window(): Pc가 현재 바라봐야하는 페이지 변경
driver.switch_to.window(driver.window_handles[1])

print("새 창으로 전환 완료")
print(f"현재 창 제목: {driver.title}")

현재 열린 창의 개수: 2
창 0: 3C1788F6A7B0F1A76394C273E2F90D1D
창 1: 4119B723D19FDAFA51F25D6566B8C443
새 창으로 전환 완료
현재 창 제목: 내보내기


 ## 8. Excel 저장 형식 선택



 팝업 창에서 Excel 형식을 선택합니다.

In [8]:
# Excel 저장 라디오 버튼이 클릭 가능할 때까지 대기
excel_button = wait.until(
    EC.element_to_be_clickable((By.XPATH,
        '//*[@id="wrap"]/form/div/div[2]/div[1]/div/ul/li[3]/label'))
)

# Excel 버튼 클릭
excel_button.click()

print("Excel 형식 선택 완료")

# 선택이 완료될 때까지 짧은 대기
time.sleep(1)

Excel 형식 선택 완료


 ## 9. 내보내기 실행 (파일 다운로드)



 **실습 포인트:**

 - JavaScript 함수를 통한 폼 제출

 - 자동 파일 다운로드

In [9]:
# RISS 웹사이트의 f_submit() JavaScript 함수 실행
# 이 함수는 선택한 형식으로 논문 목록을 다운로드합니다
driver.execute_script("f_submit()")

print("다운로드 시작!")

# 파일이 다운로드될 때까지 대기
time.sleep(3)

print(f"파일 저장 경로: {download_folder}")

다운로드 시작!
파일 저장 경로: /Users/t2023-m0099/Documents/NBC_Data_9th/설무아T/데이터 수집/4회차세션/논문목록


 ## 10. 팝업 창 닫기 및 메인 창으로 복귀

In [10]:
# 현재 창(팝업 창) 닫기
driver.close()

print("팝업 창 닫기 완료")

# 메인 창으로 전환
# window_handles[0]이 메인 검색 페이지
driver.switch_to.window(driver.window_handles[0])

print("메인 창으로 복귀 완료")
print(f"현재 창 제목: {driver.title}")

# 다음 작업을 위한 대기
time.sleep(2)

팝업 창 닫기 완료
메인 창으로 복귀 완료
현재 창 제목: RISS 검색 - 국내학술지논문


 ## 11. 다운로드된 파일 확인

In [11]:
# 다운로드 폴더의 파일 목록 확인
downloaded_files = os.listdir(download_folder)

print(f"\n{'='*80}")
print(f"[다운로드 폴더 내 파일 목록]")
print(f"{'='*80}")

if downloaded_files:
    for i, file in enumerate(downloaded_files, 1):
        file_path = os.path.join(download_folder, file)
        file_size = os.path.getsize(file_path)
        print(f"{i}. {file} ({file_size:,} bytes)")
else:
    print("다운로드된 파일이 없습니다.")

print(f"{'='*80}")


[다운로드 폴더 내 파일 목록]
1. exportExcelData_20251029205600.xls (30,720 bytes)


 ## 12. 여러 페이지 반복 다운로드 (통합 함수)

In [12]:
def download_riss_papers(driver, search_keyword, start_count):
    """
    RISS에서 논문 목록을 다운로드하는 함수
    
    Args:
        driver: Selenium WebDriver 객체
        search_keyword: 검색 키워드
        start_count: 시작 인덱스 (0, 100, 200, 300, ...)
    
    Returns:
        bool: 성공 여부
    """
    try:
        # 1. URL 생성 및 페이지 접속
        url = create_riss_url(search_keyword, start_count)
        driver.get(url)
        time.sleep(2)
        
        # 2. 전체 선택 체크박스 클릭
        select_all_checkbox = wait.until(
            EC.presence_of_element_located((By.XPATH, 
                '//*[@id="divContent"]/div/div[2]/div/div[3]/div[1]/div[1]/label/span'))
        )
        select_all_checkbox.click()
        time.sleep(1)
        
        # 3. 내보내기 버튼 클릭 (JavaScript 함수 실행)
        driver.execute_script("exportData()")
        time.sleep(2)
        
        # 4. 새 창(팝업)으로 전환
        driver.switch_to.window(driver.window_handles[1])
        
        # 5. Excel 저장 형식 선택
        excel_button = wait.until(
            EC.element_to_be_clickable((By.XPATH,
                '//*[@id="wrap"]/form/div/div[2]/div[1]/div/ul/li[3]/label'))
        )
        excel_button.click()
        time.sleep(1)
        
        # 6. 내보내기 실행 (다운로드)
        driver.execute_script("f_submit()")
        time.sleep(3)
        
        # 7. 팝업 창 닫기
        driver.close()
        
        # 8. 메인 창으로 복귀
        driver.switch_to.window(driver.window_handles[0])
        time.sleep(2)
        
        return True
        
    except Exception as e:
        print(f"오류 발생: {e}")
        
        # 오류 발생 시 메인 창으로 복귀 시도
        try:
            if len(driver.window_handles) > 1:
                driver.close()
            driver.switch_to.window(driver.window_handles[0])
        except:
            pass
        
        return False

print("논문 다운로드 함수 정의 완료!")

논문 다운로드 함수 정의 완료!


 ## 13. 여러 페이지 반복 다운로드 실행



 **실습 포인트:**

 - `range()` 함수로 페이지 인덱스 생성

 - 0, 100, 200, 300, 400 순으로 페이지 다운로드

 - 총 5페이지 = 500개 논문 목록

In [13]:
# 검색 키워드
search_term = "인공지능"

# 다운로드 통계
total_pages = 0
success_count = 0
fail_count = 0

print(f"\n{'='*80}")
print(f"[RISS 논문 목록 다운로드 시작]")
print(f"검색어: {search_term}")
print(f"페이지당 논문 수: 100개")
print(f"{'='*80}\n")

# 0부터 400까지 100씩 증가 (총 5페이지)
# range(시작, 끝, 증가값)
for i in range(0, 500, 100):
    total_pages += 1
    page_number = (i // 100) + 1
    
    print(f"[{page_number}/5] 페이지 처리 중... (시작 인덱스: {i})")
    
    # 논문 목록 다운로드
    result = download_riss_papers(driver, search_term, i)
    
    if result:
        success_count += 1
        print(f"성공: {page_number}페이지 다운로드 완료\n")
    else:
        fail_count += 1
        print(f"실패: {page_number}페이지 다운로드 실패\n")

print(f"{'='*80}")
print(f"[다운로드 완료]")
print(f"총 페이지: {total_pages}개")
print(f"성공: {success_count}개")
print(f"실패: {fail_count}개")
print(f"저장 경로: {download_folder}")
print(f"{'='*80}")


[RISS 논문 목록 다운로드 시작]
검색어: 인공지능
페이지당 논문 수: 100개

[1/5] 페이지 처리 중... (시작 인덱스: 0)
성공: 1페이지 다운로드 완료

[2/5] 페이지 처리 중... (시작 인덱스: 100)
성공: 2페이지 다운로드 완료

[3/5] 페이지 처리 중... (시작 인덱스: 200)
성공: 3페이지 다운로드 완료

[4/5] 페이지 처리 중... (시작 인덱스: 300)
성공: 4페이지 다운로드 완료

[5/5] 페이지 처리 중... (시작 인덱스: 400)
성공: 5페이지 다운로드 완료

[다운로드 완료]
총 페이지: 5개
성공: 5개
실패: 0개
저장 경로: /Users/t2023-m0099/Documents/NBC_Data_9th/설무아T/데이터 수집/4회차세션/논문목록


 ## 14. 최종 다운로드 파일 확인

In [14]:
# 다운로드 폴더의 파일 목록 확인
downloaded_files = os.listdir(download_folder)

print(f"\n{'='*80}")
print(f"[최종 다운로드 파일 목록]")
print(f"총 {len(downloaded_files)}개 파일")
print(f"{'='*80}")

if downloaded_files:
    # 파일을 최신순으로 정렬
    downloaded_files.sort(key=lambda x: os.path.getmtime(os.path.join(download_folder, x)), reverse=True)
    
    for i, file in enumerate(downloaded_files, 1):
        file_path = os.path.join(download_folder, file)
        file_size = os.path.getsize(file_path)
        # 파일 크기를 KB 단위로 변환
        file_size_kb = file_size / 1024
        print(f"{i}. {file} ({file_size_kb:.2f} KB)")
else:
    print("다운로드된 파일이 없습니다.")

print(f"{'='*80}")


[최종 다운로드 파일 목록]
총 6개 파일
1. exportExcelData_20251029205704.xls (38.00 KB)
2. exportExcelData_20251029205651.xls (32.00 KB)
3. exportExcelData_20251029205638.xls (23.50 KB)
4. exportExcelData_20251029205626.xls (28.50 KB)
5. exportExcelData_20251029205613.xls (30.00 KB)
6. exportExcelData_20251029205600.xls (30.00 KB)


 ## 15. 브라우저 종료

In [15]:
driver.quit()
print("브라우저를 종료했습니다.")

브라우저를 종료했습니다.


# 보너스 자료! 엑셀 자료 읽기 및 통합

In [ ]:
# pip install xlrd

  Using cached xlrd-2.0.2-py2.py3-none-any.whl.metadata (3.5 kB)
Using cached xlrd-2.0.2-py2.py3-none-any.whl (96 kB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 필수 라이브러리 설치
# !pip install pandas xlrd openpyxl

In [25]:
import pandas as pd
import glob

# 엑셀 파일 찾기
excel_files = glob.glob(os.path.join(download_folder, '*.xls'))

print(f"엑셀 파일 {len(excel_files)}개 발견")

# 모든 엑셀 파일 읽어서 하나로 합치기
df_list = []

for file in excel_files:
    df = pd.read_excel(file)
    
    # Unnamed 컬럼 제거
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])
    
    df_list.append(df)
    print(f"{os.path.basename(file)} - {len(df)}개 논문")

# 하나로 합치기
df_combined = pd.concat(df_list, ignore_index=True)

# 중복 제거 (제목 기준)
df_final = df_combined.drop_duplicates(subset=['제목'], keep='first')

print(f"\n총 {len(df_combined)}개 → 중복 제거 후 {len(df_final)}개")


# 데이터 미리보기
print("\n[상위 5개 논문]")
display(df_final.head())

# JSON으로 저장
json_file = os.path.join(download_folder, f'{search_term}_논문목록.json')
df_final.to_json(json_file, orient='records', force_ascii=False, indent=2)

엑셀 파일 6개 발견
exportExcelData_20251029205626.xls - 100개 논문
exportExcelData_20251029205651.xls - 100개 논문
exportExcelData_20251029205704.xls - 100개 논문
exportExcelData_20251029205600.xls - 100개 논문
exportExcelData_20251029205638.xls - 100개 논문
exportExcelData_20251029205613.xls - 100개 논문

총 600개 → 중복 제거 후 490개

[상위 5개 논문]


,번호,제목,저자,출판사,출판일
0,1,인공지능 계산기 앱 제작을 통한 수학 흥미도 향상,서강식,한국인공지능교육학회,2022
1,2,DeepAI 인공지능 개발 도구,이세훈,한국인공지능교육학회,2020
2,3,인공지능 원리-의사결정트리 놀이 수업 구상,권수용,한국인공지능교육학회,2022
3,4,인공지능 학습용 감정 데이터셋의 어휘 개체명 분석 = Analysis of Name...,이기성 ( Lee Ki Seong ),중앙대학교 인문콘텐츠연구소,2024
4,5,2022 개정 교육과정 분석을 통한 초등학교 인공지능윤리 교육 프로그램 개발,문상필,한국인공지능교육학회,2022
